# Market Basket Analysis with Apriori and FP-Growth

**A practical data mining project using supermarket transaction data**

## 1. Project Overview

Market Basket Analysis is a data mining technique used to discover products that frequently occur together in customer transactions. These patterns can be used to support decisions such as product placement, inventory planning, and product-bundle design.

This notebook explores a supermarket transaction dataset and applies two frequent-itemset mining algorithms:

- **Apriori**, which generates candidate itemsets level by level.
- **FP-Growth**, which mines frequent patterns using an FP-Tree structure.

Association rules are then generated and evaluated using **support**, **confidence**, and **lift**. A simple recommendation function is also implemented.

### Project objectives

1. Inspect and prepare transactional data.
2. Convert transactions into a binary basket matrix.
3. Mine frequent itemsets with Apriori.
4. Compare Apriori and FP-Growth.
5. Generate and interpret association rules.
6. Visualize the rules and build a basic product recommender.

> **Reproducibility note:** Run the notebook from top to bottom. Execution times may vary depending on the computer and software environment.

## 2. Environment Setup

The project uses Python and the following libraries:

- `pandas` and `numpy` for data manipulation.
- `matplotlib` for visualizations.
- `mlxtend` for Apriori, FP-Growth, and association-rule mining.

In [ ]:
# Install mlxtend when it is not already available in the environment.
%pip -q install mlxtend

# Import the libraries required for the analysis.
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

# Make displayed tables easier to read.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 3. Load the Dataset

The source file contains one row for each product recorded in a transaction.

Expected columns:

- `Transaction_ID`: identifier of a transaction.
- `Produit`: product recorded in that transaction.

The repository stores the CSV file in the `data/` directory. If the notebook is opened directly in Google Colab and the file is not found, the next cell asks you to upload the CSV file.

In [ ]:
# Locate the dataset in the repository.
DATA_PATH = Path("data/transactions_supermarche.csv")

# Support running the notebook when the CSV is uploaded beside the notebook.
if not DATA_PATH.exists():
    DATA_PATH = Path("transactions_supermarche.csv")

# If the file is still missing, allow the user to upload it in Google Colab.
if not DATA_PATH.exists():
    try:
        from google.colab import files
        uploaded_files = files.upload()
        if uploaded_files:
            DATA_PATH = Path(next(iter(uploaded_files)))
    except ImportError:
        raise FileNotFoundError(
            "Dataset not found. Place transactions_supermarche.csv in the data/ folder."
        )

# Load the transaction data.
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
df.head(10)

## 4. Exploratory Data Analysis

This section checks the dataset structure and basic data quality. It also identifies the most frequently recorded products.

In [ ]:
# Inspect the dataset schema and data types.
print("Dataset information:")
df.info()

# Calculate basic dataset statistics.
number_of_transactions = df["Transaction_ID"].nunique()
number_of_products = df["Produit"].nunique()
number_of_records = len(df)

print("\nNumber of transactions:", number_of_transactions)
print("Number of unique products:", number_of_products)
print("Number of transaction-product records:", number_of_records)

# Check for missing values and exact duplicate rows.
print("\nMissing values by column:")
print(df.isna().sum())

print("\nNumber of duplicate rows:", df.duplicated().sum())

# Display the ten most frequently recorded products.
top_products = df["Produit"].value_counts().head(10)
print("\nTop 10 most frequently recorded products:")
display(top_products.to_frame(name="Purchase records"))

# Visualize the ten most frequently recorded products.
plt.figure(figsize=(10, 5))
top_products.plot(kind="bar")
plt.title("Top 10 Most Frequently Recorded Products")
plt.xlabel("Product")
plt.ylabel("Number of transaction records")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Exploratory Analysis — Interpretation

The dataset contains **500 transactions**, **20 unique products**, and **1,968 transaction-product records**. The two columns required for this analysis are present. The dataset has no missing values and no exact duplicate rows.

The product-frequency chart shows how often each product appears in the transaction records. This frequency is useful for understanding the dataset, but it does not by itself identify products that are purchased together. Association-rule mining is used for that purpose.

## 5. Data Preparation: Build the Basket Matrix

Association-rule algorithms require a transaction-product matrix. Each row represents one transaction and each column represents one product.

- `True` means that the product is present in the transaction.
- `False` means that the product is absent from the transaction.

In [ ]:
# Convert the long transaction table into a transaction-product matrix.
basket = pd.crosstab(
    df["Transaction_ID"],
    df["Produit"]
).astype(bool)

print("First five transactions:")
display(basket.head())

print("Basket matrix dimensions:", basket.shape)
print("Value data type:", basket.values.dtype)

# Count the number of products in each transaction.
basket_size = basket.sum(axis=1)

print("\nAverage basket size:", round(basket_size.mean(), 3))
print("\nDistribution of basket sizes:")
display(
    basket_size.value_counts()
    .sort_index()
    .rename_axis("Number of products")
    .to_frame("Number of transactions")
)

### Data Preparation — Interpretation

The basket matrix contains **500 rows** and **20 product columns**. Each row represents one transaction, and Boolean values indicate whether a product is present.

The average basket contains approximately **3.94 products**. The matrix is now in the format required by the Apriori and FP-Growth algorithms.

## 6. Frequent Itemset Mining with Apriori

An itemset is a group of one or more products. A frequent itemset is an itemset whose support is at least the selected minimum support threshold.

For this project, the initial minimum support is **0.05**. With 500 transactions, this means that an itemset must appear in at least 25 transactions to be considered frequent.

In [ ]:
# Define the minimum support threshold.
MIN_SUPPORT = 0.05

# Apply Apriori to identify frequent itemsets.
frequent_itemsets = apriori(
    basket,
    min_support=MIN_SUPPORT,
    use_colnames=True
)

# Sort itemsets from highest to lowest support.
frequent_itemsets = frequent_itemsets.sort_values(
    by="support",
    ascending=False
).reset_index(drop=True)

print("Number of frequent itemsets:", len(frequent_itemsets))
display(frequent_itemsets)

In [ ]:
# Separate frequent itemsets containing exactly two and three products.
itemsets_2 = frequent_itemsets[
    frequent_itemsets["itemsets"].apply(len) == 2
]

itemsets_3 = frequent_itemsets[
    frequent_itemsets["itemsets"].apply(len) == 3
]

print("Frequent itemsets of size 2:", len(itemsets_2))
display(itemsets_2)

print("\nFrequent itemsets of size 3:", len(itemsets_3))
display(itemsets_3)

In [ ]:
# Compare the number of frequent itemsets at different support thresholds.
support_thresholds = [0.05, 0.10, 0.20]
support_comparison = []

for threshold in support_thresholds:
    itemsets_at_threshold = apriori(
        basket,
        min_support=threshold,
        use_colnames=True
    )

    support_comparison.append({
        "Minimum support": threshold,
        "Total itemsets": len(itemsets_at_threshold),
        "Itemsets of size 2": (
            itemsets_at_threshold["itemsets"].apply(len) == 2
        ).sum(),
        "Itemsets of size 3": (
            itemsets_at_threshold["itemsets"].apply(len) == 3
        ).sum()
    })

support_comparison = pd.DataFrame(support_comparison)
display(support_comparison)

### Apriori — Interpretation

A lower minimum support threshold generally allows more itemsets to be retained, including less frequent combinations. Increasing the threshold makes the selection stricter and removes itemsets that occur in fewer transactions.

The support threshold should be selected according to the purpose of the analysis: a very low threshold can produce many less common patterns, while a high threshold can exclude potentially useful combinations.

## 7. Association Rule Mining

An association rule has the form **A → B**, where A is the antecedent and B is the consequent.

Three metrics are used:

- **Support:** proportion of all transactions containing both A and B.
- **Confidence:** proportion of transactions containing A that also contain B.
- **Lift:** confidence divided by the overall support of B.

Interpretation of lift:

- `Lift > 1`: positive association.
- `Lift = 1`: the two sides are independent according to this measure.
- `Lift < 1`: negative association.

A high confidence value alone does not necessarily indicate a strong association. Lift compares confidence with how common the consequent is overall.

In [ ]:
# Recalculate frequent itemsets at the selected support threshold.
# This ensures the association rules use the intended itemset collection.
frequent_itemsets = apriori(
    basket,
    min_support=MIN_SUPPORT,
    use_colnames=True
)

# Generate rules with a minimum confidence of 0.50.
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.50
)

# Keep the metrics required for this analysis.
rules = rules[
    ["antecedents", "consequents", "support", "confidence", "lift"]
].copy()

# Sort rules by descending lift.
rules = rules.sort_values(
    by="lift",
    ascending=False
).reset_index(drop=True)

print("Number of association rules:", len(rules))
display(rules)

# Keep rules showing a positive association (lift greater than 1).
rules_filtered = rules[rules["lift"] > 1].copy()

print("\nNumber of rules with lift > 1:", len(rules_filtered))
display(rules_filtered)

In [ ]:
# Manually calculate the metrics for the highest-lift filtered rule.
if not rules_filtered.empty:
    selected_rule = rules_filtered.iloc[0]

    antecedent = selected_rule["antecedents"]
    consequent = selected_rule["consequents"]

    # Identify transactions containing every product in the antecedent.
    contains_antecedent = basket[list(antecedent)].all(axis=1)

    # Identify transactions containing every product in the consequent.
    contains_consequent = basket[list(consequent)].all(axis=1)

    # Identify transactions containing both sides of the rule.
    contains_both = contains_antecedent & contains_consequent

    # Calculate support values from Boolean transaction indicators.
    support_antecedent = contains_antecedent.mean()
    support_consequent = contains_consequent.mean()
    support_both = contains_both.mean()

    # Calculate confidence and lift manually.
    manual_support = support_both
    manual_confidence = support_both / support_antecedent
    manual_lift = manual_confidence / support_consequent

    print("Selected rule:", antecedent, "->", consequent)
    print("Manual support:", manual_support)
    print("Manual confidence:", manual_confidence)
    print("Manual lift:", manual_lift)

    print("\nMetrics returned by association_rules:")
    print("Support:", selected_rule["support"])
    print("Confidence:", selected_rule["confidence"])
    print("Lift:", selected_rule["lift"])
else:
    print("No rules with lift greater than 1 were found.")

### Association Rules — Interpretation

The rules are sorted by lift so that the strongest positive associations appear first. For example, the dataset includes the rule **Pates → Sauce_tomate**, with support of **13%**, confidence of approximately **73.03%**, and lift of approximately **4.87**.

This means that 13% of all transactions contain both products, and approximately 73.03% of transactions containing pasta also contain tomato sauce. The lift indicates that the consequent occurs much more often when the antecedent is present than it does overall.

These metrics describe associations in the observed dataset; they do not establish that purchasing one product causes the purchase of another.

## 8. Rule Visualization and Business Interpretation

The scatter plot displays support on the horizontal axis and confidence on the vertical axis. Point color represents lift.

In [ ]:
# Visualize the association rules using support, confidence, and lift.
if not rules_filtered.empty:
    plt.figure(figsize=(9, 6))

    scatter = plt.scatter(
        rules_filtered["support"],
        rules_filtered["confidence"],
        c=rules_filtered["lift"],
        cmap="viridis",
        alpha=0.8
    )

    plt.xlabel("Support")
    plt.ylabel("Confidence")
    plt.title("Association Rules: Support vs. Confidence")
    plt.colorbar(scatter, label="Lift")
    plt.tight_layout()
    plt.show()
else:
    print("No filtered rules are available to visualize.")

In [ ]:
# Display rules with confidence greater than 0.70.
rules_high_confidence = rules_filtered[
    rules_filtered["confidence"] > 0.70
].sort_values("lift", ascending=False)

print("Rules with confidence greater than 0.70:")
display(rules_high_confidence)

### Visualization and Business Interpretation

The rules with lift greater than 1 represent positive associations in this dataset. Several examples may support practical retail analysis:

1. **Product bundles:** the association Pates → Sauce_tomate may be considered when evaluating a pasta-and-sauce bundle.
2. **Product placement:** the association Biscuits → Lait may be considered when reviewing the placement of these products.
3. **Inventory monitoring:** the association Beurre → Pain may be useful when monitoring the availability of complementary products.

These are hypotheses for business decisions, not guaranteed improvements. The store should validate them using sales, margin, stock, and customer data. Association rules do not prove causation.

## 9. Simple Product Recommendation Function

The function below recommends consequents from rules whose antecedent is exactly the selected single product. It ranks recommendations by lift and returns up to `top_n` distinct products.

In [ ]:
def recommend(product, rules_table, top_n=5):
    """Recommend products using rules with an exact single-product antecedent.

    Parameters
    ----------
    product : str
        Product used as the recommendation input.
    rules_table : pandas.DataFrame
        Association rules containing antecedents, consequents,
        support, confidence, and lift.
    top_n : int, default=5
        Maximum number of recommendations to return.

    Returns
    -------
    pandas.DataFrame
        Recommended products ranked by descending lift.
    """
    recommendations = []

    for _, rule in rules_table.iterrows():
        # Match only rules whose antecedent is exactly this one product.
        if rule["antecedents"] == frozenset([product]):
            for recommended_product in rule["consequents"]:
                recommendations.append({
                    "Recommended product": recommended_product,
                    "Support": rule["support"],
                    "Confidence": rule["confidence"],
                    "Lift": rule["lift"]
                })

    result = pd.DataFrame(recommendations)

    if result.empty:
        return result

    # Keep the strongest rule for each recommended product.
    result = result.sort_values(
        by="Lift",
        ascending=False
    ).drop_duplicates(subset="Recommended product")

    return result.head(top_n)


# Example: find recommendations for pasta.
pasta_recommendations = recommend(
    "Pates",
    rules_filtered,
    top_n=5
)

display(pasta_recommendations)

### Recommendation Function — Interpretation

For the input product **Pates**, the function returns **Sauce_tomate** as a recommendation in the observed rules. The corresponding rule has support of **13%**, confidence of approximately **73.03%**, and lift of approximately **4.87**.

The function is intentionally limited to rules with an exact single-product antecedent. Rules requiring multiple products are not treated as recommendations from one product alone.

## 10. Additional Experiments

### 10.1 Apriori vs. FP-Growth

Both algorithms identify frequent itemsets, but they use different mining strategies:

- **Apriori** generates candidate itemsets level by level and uses the Apriori property to prune candidates.
- **FP-Growth** compresses transactions into an FP-Tree and mines frequent patterns without explicitly generating the same candidate sets.

The following experiment compares their execution times and verifies that they return the same itemsets and support values.

In [ ]:
# Use the same support threshold for a fair comparison.
comparison_support = 0.05

# Measure Apriori execution time.
start_time = perf_counter()
itemsets_apriori = apriori(
    basket,
    min_support=comparison_support,
    use_colnames=True
)
apriori_time = perf_counter() - start_time

# Measure FP-Growth execution time.
start_time = perf_counter()
itemsets_fpgrowth = fpgrowth(
    basket,
    min_support=comparison_support,
    use_colnames=True
)
fpgrowth_time = perf_counter() - start_time

# Convert the outputs into dictionaries to compare itemsets and supports.
apriori_supports = {
    frozenset(row.itemsets): row.support
    for row in itemsets_apriori.itertuples(index=False)
}
fpgrowth_supports = {
    frozenset(row.itemsets): row.support
    for row in itemsets_fpgrowth.itertuples(index=False)
}

same_itemsets = apriori_supports.keys() == fpgrowth_supports.keys()
same_supports = (
    same_itemsets
    and all(
        np.isclose(apriori_supports[itemset], fpgrowth_supports[itemset])
        for itemset in apriori_supports
    )
)

comparison_algorithms = pd.DataFrame([
    {
        "Algorithm": "Apriori",
        "Frequent itemsets": len(itemsets_apriori),
        "Execution time (seconds)": apriori_time
    },
    {
        "Algorithm": "FP-Growth",
        "Frequent itemsets": len(itemsets_fpgrowth),
        "Execution time (seconds)": fpgrowth_time
    }
])

display(comparison_algorithms)
print("Same frequent itemsets:", same_itemsets)
print("Same support values:", same_supports)

### Apriori vs. FP-Growth — Interpretation

With a minimum support of 0.05, both algorithms are expected to identify the same frequent itemsets and support values. Their execution times can differ because they use different mining strategies.

The faster algorithm in one run is not necessarily faster on every dataset. Runtime depends on dataset size, transaction density, hardware, software versions, and implementation details. In this experiment, compare the measured times from your own execution rather than treating a previous run's timing as a fixed result.

### 10.2 Sensitivity to Confidence Thresholds

This experiment evaluates how the minimum confidence threshold affects the number of generated association rules. The same frequent itemsets are used for all thresholds so that the comparison remains consistent.

In [ ]:
# Reuse the frequent itemsets generated with minimum support 0.05.
confidence_thresholds = [0.30, 0.50, 0.70]
confidence_results = []

for threshold in confidence_thresholds:
    rules_at_threshold = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=threshold
    )

    confidence_results.append({
        "Minimum confidence": threshold,
        "Number of rules": len(rules_at_threshold)
    })

confidence_comparison = pd.DataFrame(confidence_results)
display(confidence_comparison)

### Confidence Thresholds — Interpretation

Increasing the minimum confidence threshold generally reduces the number of rules because fewer rules satisfy the stricter condition. A lower threshold returns more rules, but some may have weaker conditional reliability.

The selected threshold should balance the number of rules with their usefulness. Confidence should always be considered together with support and lift.

### 10.3 Rules with Multiple Antecedents

Some rules require more than one product on the left-hand side. These rules describe combinations of products associated with another product.

In [ ]:
# Select rules whose antecedent contains at least two products.
rules_multiple_antecedents = rules[
    rules["antecedents"].apply(len) >= 2
].copy()

rules_multiple_antecedents = rules_multiple_antecedents.sort_values(
    by="lift",
    ascending=False
)

print("Number of rules with multiple antecedents:",
      len(rules_multiple_antecedents))
display(rules_multiple_antecedents)

### Multiple Antecedents — Interpretation

A rule such as **(Sucre, Pain) → Cafe** describes an association where the combination of sugar and bread appears with coffee. Such rules can support more specific analysis than rules based on a single antecedent.

However, multi-product rules may have lower support because fewer transactions contain the complete combination. A confidence of 100% in a limited dataset should also be interpreted carefully; it does not guarantee the same result in future transactions.

### 10.4 Summary of the Most Relevant Rules

The following table displays up to five rules with the highest lift among the rules whose lift is greater than 1.

In [ ]:
# Display the five highest-lift rules for a concise project summary.
top_rules = rules_filtered.sort_values(
    by="lift",
    ascending=False
).head(5)

display(top_rules)

## 11. Conclusion

This project applied Market Basket Analysis to a supermarket transaction dataset containing 500 transactions and 20 unique products. The transaction records were transformed into a Boolean basket matrix, and frequent itemsets were mined using Apriori and FP-Growth.

Association rules were evaluated using support, confidence, and lift. Examples such as **Pates → Sauce_tomate**, **Sucre → Cafe**, and **Biscuits → Lait** illustrate positive associations in the observed transactions. A simple recommendation function was implemented to suggest a product from a single-product antecedent.

The results may help inform product-bundle ideas, product placement, and inventory monitoring. However, the dataset is limited in size and association does not imply causation. Further analysis with larger datasets, time periods, prices, and profit margins would be needed before making operational decisions.

## 12. Reproducibility and Dataset Information

To reproduce this analysis:

1. Install the packages listed in `requirements.txt`.
2. Keep `transactions_supermarche.csv` inside the repository's `data/` directory.
3. Open this notebook and run all cells from top to bottom.

The dataset uses the columns `Transaction_ID` and `Produit`. The CSV file is included in this project folder for reproducibility. Before making the repository public, ensure that you have permission to share the dataset.